# Getting Started with dPASP

dPASP is a framework for probabilistic answer set programming. It lets you write logic programs with uncertain (probabilistic) facts and ask queries about the probability of events.

This notebook walks through the basics:
1. Installing dPASP
2. Parsing and inspecting a program
3. Running inference and reading results
4. Writing programs inline as strings
5. Conditional queries (evidence)
6. Working with results in Python

## 1. Installation

Install dPASP from PyPI:

```bash
pip install pasp-plp-uas
```

This will compile the C extensions automatically. You'll need a C compiler (e.g. `clang` or `gcc`) available on your system.

In [9]:
import pasp
import numpy as np

## 2. Parsing a program from a `.plp` file

A `.plp` file describes probabilistic facts, logic rules, and queries.

Here's what `simple.plp` looks like:
```
0.8::r. 0.2::s(a). 0.3::s(b).
v :- r, s(a), s(b).

#query(s(a))
#query(s(b))
#query(v)
```

This says: `r` is true with probability 0.8, `s(a)` with 0.2, `s(b)` with 0.3. The rule says `v` is true when all three hold. We then query the probability of `s(a)`, `s(b)`, and `v`.

`pasp.parse()` reads the file and returns a `Program` object.

In [10]:
P = pasp.parse("simple.plp")
print(P)

<Logic Program:
v :- r, s(a), s(b).,
Probabilistic Facts:
[0.8::r, 0.2::s(a), 0.3::s(b)],
Queries:
[ℙ(s(a)), ℙ(s(b)), ℙ(v)]>


## 3. Running inference

Call the program to run inference. Use `quiet=True` to suppress the C-level terminal output and get a clean result object back.

In [11]:
result = P(quiet=True)
print(result)

ℙ(s(a)) = [0.200000, 0.200000]
ℙ(s(b)) = [0.300000, 0.300000]
ℙ(v) = [0.000000, 0.000000]


Querying:                                                               0h00m00s


Each line shows a query and its probability interval `[lower, upper]`. When lower equals upper, the probability is exact.

## 4. Writing programs inline

Instead of a file, you can pass a program string directly with `from_str=True`.

Here's the classic insomnia example: you either sleep or work, but insomnia can prevent sleep.

In [12]:
insomnia_program = """
0.3::insomnia.
sleep :- not work, not insomnia.
work :- not sleep.

#query(insomnia)
#query(work)
#query(sleep)
"""

P2 = pasp.parse(insomnia_program, from_str=True)
result2 = P2(quiet=True)
print(result2)

ℙ(insomnia) = [0.300000, 0.300000]
ℙ(work) = [0.000000, 0.000000]
ℙ(sleep) = [0.000000, 0.000000]


Querying:                                                               0h00m00s


## 5. Conditional queries (evidence)

You can query the probability of an event *given* that some evidence holds, using the `|` syntax.

The earthquake/burglary example models an alarm that can be triggered by a burglary or an earthquake, and asks: what is the probability of the alarm given different combinations of evidence?

In [13]:
P3 = pasp.parse("earthquake.plp")
result3 = P3(quiet=True)
print(result3)

ℙ(alarm | burglary, earthquake) = [0.000000, 0.000000]
ℙ(alarm | not burglary, earthquake) = [0.000000, 0.000000]
ℙ(alarm | burglary, not earthquake) = [0.000000, 0.000000]
ℙ(alarm | not burglary, not earthquake) = [0.000000, 0.000000]


Querying:                                                               0h00m00s


## 6. Working with results in Python

The result object wraps a numpy array, so you can use it in computations directly.

### String representation

`str(result)` gives the formatted output, useful for logging or display.

In [14]:
result_str = str(result)
print(repr(result_str))

'ℙ(s(a)) = [0.200000, 0.200000]\nℙ(s(b)) = [0.300000, 0.300000]\nℙ(v) = [0.000000, 0.000000]'


### Raw numpy data

`result.data` is the underlying array with shape `(num_queries, 2)` -- each row is `[lower, upper]`.

In [15]:
result.data

array([[0.2, 0.2],
       [0.3, 0.3],
       [0. , 0. ]])

In [16]:
result.data.shape

(3, 2)

### Indexing and iteration

You can index into the result or iterate over it alongside the query names.

In [17]:
# First query's probability interval
result[0]

array([0.2, 0.2])

In [18]:
for query, probs in zip(result.queries, result):
    print(f"{query}  ->  lower={probs[0]:.4f}, upper={probs[1]:.4f}")

ℙ(s(a))  ->  lower=0.2000, upper=0.2000
ℙ(s(b))  ->  lower=0.3000, upper=0.3000
ℙ(v)  ->  lower=0.0000, upper=0.0000


### Numpy interop

The result can be passed directly to numpy functions.

In [19]:
arr = np.asarray(result)
print("type:", type(arr))
print("mean per column (lower, upper):", arr.mean(axis=0))

type: <class 'numpy.ndarray'>
mean per column (lower, upper): [0.16666667 0.16666667]
